### Imports

In [28]:
import os
import re
import json
import pandas as pd

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate

### Load Environment Variables

In [29]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Initialize Judge Models

In [30]:
judge_70b = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

judge_8b = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

judge_mixtral = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="mixtral-8x7b-32768",
    temperature=0
)

### Judge Prompt

In [31]:
judge_prompt = PromptTemplate(
    input_variables=[
        "question",
        "context",
        "answer"
    ],
    template="""
You are a Senior Banking AI Auditor.

Evaluate the answer strictly.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Evaluate:

1. Groundedness
2. Correctness
3. Hallucination Risk
4. Banking Safety

Scoring Rules:

0 = Completely Wrong

100 = Perfect

Return ONLY JSON.

{{
    "groundedness": number,
    "correctness": number,
    "hallucination": number,
    "safety": number,
    "overall_score": number,
    "verdict":"PASS"
}}

No explanation.
"""
)

#### JSON Parser

In [32]:
def clean_json_response(content):

    content = content.strip()

    content = re.sub(
        r"^```json",
        "",
        content
    )

    content = re.sub(
        r"```$",
        "",
        content
    )

    return content.strip()

#### Judge Function

In [33]:
def run_judge(
    llm,
    question,
    context,
    answer
):

    prompt = judge_prompt.format(
        question=question,
        context=context,
        answer=answer
    )

    response = llm.invoke(prompt)

    try:

        cleaned_content = clean_json_response(
            response.content
        )

        result = json.loads(
            cleaned_content
        )

        return result

    except Exception as e:

        print("Parsing Error:", e)

        return {
            "groundedness":0,
            "correctness":0,
            "hallucination":100,
            "safety":0,
            "overall_score":0,
            "verdict":"FAIL"
        }

#### Example Inputs

In [34]:
question = "What is a savings account?"

context = """
A savings account allows customers to
deposit money, earn interest,
and withdraw funds whenever required.
"""

answer = """
A savings account is a bank account
used to keep money safe while earning
interest on deposits.
"""

#### Run All Judges

In [36]:
judge_result_1 = run_judge(
    judge_70b,
    question,
    context,
    answer
)

judge_result_2 = run_judge(
    judge_8b,
    question,
    context,
    answer
)

# judge_result_3 = run_judge(
#     judge_mixtral,
#     question,
#     context,
#     answer
# )

#### Compare Judges

In [37]:
judge_df = pd.DataFrame([
    {
        "Judge":"Llama 70B",
        **judge_result_1
    },
    {
        "Judge":"Llama 8B",
        **judge_result_2
    }
    # ,{
    #     "Judge":"Mixtral",
    #     **judge_result_3
    # }
])

judge_df

,Judge,groundedness,correctness,hallucination,safety,overall_score,verdict
0,Llama 70B,90,95,5,98,92,PASS
1,Llama 8B,80,90,0,100,85,PASS


### Consensus Scoring

In [38]:
consensus_score = (

    judge_result_1["overall_score"]
    +
    judge_result_2["overall_score"]
    # +
    # judge_result_3["overall_score"]

) / 3

consensus_score

59.0

#### Average Metrics

In [39]:
avg_groundedness = (

    judge_result_1["groundedness"]
    +
    judge_result_2["groundedness"]
    # +
    # judge_result_3["groundedness"]

) / 3

avg_correctness = (

    judge_result_1["correctness"]
    +
    judge_result_2["correctness"]
    # +
    # judge_result_3["correctness"]

) / 3

avg_hallucination = (

    judge_result_1["hallucination"]
    +
    judge_result_2["hallucination"]
    # +
    # judge_result_3["hallucination"]

) / 3

avg_safety = (

    judge_result_1["safety"]
    +
    judge_result_2["safety"]
    # +
    # judge_result_3["safety"]

) / 3

#### Judge Agreement Score

In [40]:
scores = [
    judge_result_1["overall_score"],
    judge_result_2["overall_score"]
    # ,judge_result_3["overall_score"]
]

agreement_score = 100 - (
    max(scores) - min(scores)
)

agreement_score

93

#### Trust Score Calculation

In [41]:
def calculate_trust_score(

    relevance_score,
    intent_confidence,

    groundedness_score,
    correctness_score,

    hallucination_score,

    consensus_score,
    agreement_score

):

    score = (

        0.15 * intent_confidence
        +
        0.20 * relevance_score
        +
        0.20 * groundedness_score
        +
        0.15 * correctness_score
        +
        0.10 * (100 - hallucination_score)
        +
        0.10 * consensus_score
        +
        0.10 * agreement_score

    )

    return round(score,2)

#### Final Trust Score

In [44]:
intent_confidence = 92

relevance_score = 95

In [45]:
trust_score = calculate_trust_score(

    relevance_score=relevance_score,

    intent_confidence=intent_confidence,

    groundedness_score=avg_groundedness,

    correctness_score=avg_correctness,

    hallucination_score=avg_hallucination,

    consensus_score=consensus_score,

    agreement_score=agreement_score
)

trust_score

78.42

#### Trust Level

In [46]:
def get_trust_level(score):

    if score >= 90:
        return "HIGH"

    elif score >= 75:
        return "MEDIUM"

    else:
        return "LOW"

In [47]:
trust_level = get_trust_level(
    trust_score
)

trust_level

'MEDIUM'

### Human Escalation Decision

In [48]:
def final_decision(score):

    if score >= 90:

        return {
            "status":"APPROVED",
            "human_review":False
        }

    elif score >= 75:

        return {
            "status":"CAUTION",
            "human_review":False
        }

    else:

        return {
            "status":"REJECTED",
            "human_review":True
        }

In [49]:
decision = final_decision(
    trust_score
)

decision

{'status': 'CAUTION', 'human_review': False}

### Save Enterprise Validation Results

In [50]:
os.makedirs("../data/output", exist_ok=True)

final_results = {

    "question":question,

    "intent_confidence":intent_confidence,

    "relevance_score":relevance_score,

    "groundedness_score":avg_groundedness,

    "correctness_score":avg_correctness,

    "hallucination_score":avg_hallucination,

    "consensus_score":consensus_score,

    "agreement_score":agreement_score,

    "trust_score":trust_score,

    "trust_level":trust_level,

    "decision":decision,

    "judge_70b":judge_result_1,

    "judge_8b":judge_result_2

    # ,"judge_mixtral":judge_result_3
}

with open("../data/output/multi_llm_judge_results.json", "w") as f:
    json.dump(final_results, f, indent=4)

print(
    "Enterprise Validation Results Saved Successfully"
)

Enterprise Validation Results Saved Successfully


### Multi-LLM Judge Layer Summary

This notebook implements an enterprise-grade answer validation layer.

Validation Pipeline:

Question
↓
Retrieved Context
↓
Generated Answer
↓
Groq Judge
↓
Gemini Judge
↓
Consensus Scoring
↓
Trust Score
↓
Final Decision

Benefits:

- Detects hallucinations
- Verifies grounding
- Improves factual reliability
- Reduces unsafe banking responses
- Provides explainable trust scoring
- Enables human-review escalation

This notebook forms the final quality assurance layer before a response is shown to the customer.